# 강의 04 · 실습 3 — 서브그래프 모듈화 · (1) 강사 시연

## 1. 문제상황

- 온라인 쇼핑몰 고객센터의 메일 처리 그래프는 요구가 늘 때마다 노드 함수와 엣지가 같은 파일에 함께 쌓였습니다.
- 분류 부분만 따로 실행해 보려면 그래프 전체를 실행해야 합니다.
- 같은 분류 절차를 다른 팀의 그래프에서 쓰려면, 노드 함수와 연결을 그 그래프에 다시 옮겨 적어야 합니다.
- 다른 팀이 만든 분류 그래프는 상태의 키 이름이 우리 그래프와 달라서, 그대로 노드로 등록할 수 없습니다.

## 2. 문제와 목표

- **문제**: 분류 절차가 그래프 안에 흩어져 있어 따로 실행하거나 다른 그래프에서 재사용할 수 없습니다. 상태 키 이름이 다른 그래프는 노드로 등록할 방법을 모릅니다.
- **목표**
  - 분류 절차를 따로 컴파일한 자식 그래프(서브그래프)로 만들고, 부모 그래프에 노드로 등록합니다.
    - 얹는 방식 둘: 방식 A(부모와 자식이 같은 상태 스키마)는 컴파일된 자식 그래프를 노드로 직접 넣고, 방식 B(다른 상태 스키마)는 키를 번역하는 함수로 자식을 감싸서 넣습니다.
- **목표 달성 여부의 판정 기준**:
  - 같은 배송 문의 메일을 자식 그래프 단독, 방식 A 부모, 방식 B 부모에 차례로 넣었을 때,
  - 세 실행의 분류 결과가 같고, 두 부모의 실행 결과에 부모 노드 이름(classifier, handle)만 출력되는 것을 확인합니다.

## 3. 워크플로우 다이어그램

![워크플로우 다이어그램](imgs/lec04_ex03_s1_diagram.svg)

## 4. 단계별 요구사항

1. **상태를 정의합니다.**
    - 부모와 자식 A가 함께 쓰는 공용 상태 `SharedState`는 메일 본문(`email`), 분류 결과(`category`), 배정 결과(`handled`) 키 세 개를 가집니다.
    - 자식 B 전용 상태 `SubState`는 분류할 글(`text`), 분류 결과(`label`) 키 두 개를 가집니다.
2. **자식 노드 함수를 만듭니다.**
    - detect 노드는 공용 상태의 메일 본문에 한글이 있는지 규칙으로 판단해 화면에 출력하고 상태를 바꾸지 않습니다.
    - classify_shared 노드는 공용 상태의 `email`을 읽어 환불·배송·기타 중 한 단어를 `category` 키에 씁니다.
    - classify_private 노드는 자식 B 상태의 `text`를 읽어 같은 기준으로 분류한 결과를 `label` 키에 씁니다.
3. **부모 노드 함수를 만듭니다.**
    - handle 노드는 공용 상태의 분류 결과를 읽어 「<분류> 담당자에게 배정」 문장을 `handled` 키에 씁니다.
4. **자식 그래프를 구성하고 컴파일합니다.**
    - 자식 A는 공용 상태로 detect → classify를 연결하고, 자식 B는 자식 B 상태로 classify 하나를 둡니다.
    - 둘 다 컴파일해 단독으로 실행할 수 있는 그래프로 만듭니다.
5. **부모 그래프에 노드를 등록합니다.**
    - 방식 A 부모는 컴파일된 자식 A를 `add_node`의 두 번째 인자로 직접 넣습니다.
    - 방식 B 부모는 부모 상태의 `email`을 자식 B의 `text`로 넣고 자식이 돌려준 `label`을 부모의 `category`로 되받는 래퍼 함수를 만들어 넣습니다.
    - 두 부모 모두 handle 노드를 함께 등록합니다.
6. **엣지를 연결합니다.**
    - 두 부모 모두 START → classifier → handle → END를 고정 엣지로 연결합니다.
7. **그래프를 컴파일하고 실행합니다.**
    - 자식 A를 단독으로 실행한 뒤, 같은 메일을 방식 A 부모와 방식 B 부모에 넣어 노드가 하나 끝날 때마다 바뀐 키를 출력합니다.

## 5. 코드 골격 — LangGraph 5단

랭그래프(LangGraph)로 그래프를 세우는 순서는 다음 다섯 단계입니다. 자식 그래프도 같은 다섯 단계로 세우며, 컴파일된 자식 그래프를 부모의 노드로 등록하는 일은 ③ 노드 등록 단계의 확장입니다.

| 단계 | 하는 일 | 사용하는 코드 | 대응하는 요구사항 |
|---|---|---|---|
| ① 상태 정의 | 부모·자식이 쓸 상태의 키를 선언합니다 | `class SharedState(TypedDict)`, `class SubState(TypedDict)` | 1 |
| ② 노드 함수 정의 | 상태를 받아 바뀐 키만 돌려주는 함수를 만듭니다 | `def classify_shared(state) -> dict` | 2, 3 |
| ③ 그래프 빌더 생성과 노드 등록 | 자식 그래프를 세워 컴파일하고, 부모 그래프에 자식(또는 래퍼 함수)과 노드를 등록합니다 | `StateGraph(SharedState)`, `add_node("classifier", CHILD_A)` | 4, 5 |
| ④ 엣지 연결 | 부모 노드 사이의 순서를 정합니다 | `add_edge` | 6 |
| ⑤ 컴파일과 실행 | 부모 그래프를 컴파일하고 입력을 넣어 실행합니다 | `compile()`, `invoke`, `stream()` | 7 |

## 6. 코드 — 스텝바이스텝

### 단계 0 — 준비

라이브러리를 불러오고 모델을 준비합니다.

- API 키는 `.env` 파일에서 읽습니다.
- `.env` 파일은 실습 루트 폴더(`agentic-ai`)에 한 개만 둡니다. `find_dotenv()`가 노트북 위치에서 상위 폴더로 올라가며 찾습니다.
- `.env` 파일에는 다음 한 줄만 넣습니다.

```
OPENAI_API_KEY=발급받은_키
```

In [1]:
import os

from dotenv import load_dotenv, find_dotenv
from typing import TypedDict

from langchain.chat_models import init_chat_model
from langchain_core.messages import HumanMessage, SystemMessage
from langgraph.graph import END, START, StateGraph

load_dotenv(find_dotenv(usecwd=True))
if not os.environ.get("OPENAI_API_KEY"):
    raise SystemExit("agentic-ai 폴더의 .env 파일에 OPENAI_API_KEY 한 줄을 넣습니다.")

llm = init_chat_model("openai/gpt-5.6-luna", model_provider="litellm")
print("모델 준비를 마쳤습니다.")

모델 준비를 마쳤습니다.


### 단계 ① — 상태 정의 (요구사항 1)

상태 스키마를 두 개 선언합니다. `SharedState`는 부모와 자식 A가 그대로 함께 쓰고, `SubState`는 자식 B만 씁니다. 방식 A와 방식 B의 차이는 자식이 부모와 같은 스키마를 쓰는지 여부에서 시작합니다.

In [2]:
class SharedState(TypedDict):
    email: str      # 받은 메일 본문
    category: str   # 분류 결과 (환불 / 배송 / 기타)
    handled: str    # 배정 결과


class SubState(TypedDict):   # 자식 B 전용. 키 이름이 부모와 다르다
    text: str    # 분류할 글
    label: str   # 분류 결과


print("공용 상태의 키:", list(SharedState.__annotations__))
print("자식 B 상태의 키:", list(SubState.__annotations__))

공용 상태의 키: ['email', 'category', 'handled']
자식 B 상태의 키: ['text', 'label']


### 단계 ② — 노드 함수 정의 (요구사항 2, 3)

- 노드는 상태를 인자로 받아 딕셔너리를 돌려주는 파이썬 함수입니다. 자식의 노드도 부모의 노드와 같은 모양입니다.
- classify_shared와 classify_private는 하는 일이 같고 읽고 쓰는 키 이름만 다릅니다. 상태 스키마가 다르면 노드 함수도 그 스키마의 이름으로 써야 합니다.

In [3]:
def detect(state: SharedState) -> dict:
    """메일의 언어를 규칙으로 감지해 출력한다 (상태를 바꾸지 않는다)."""
    lang = "한국어" if any("가" <= ch <= "힣" for ch in state["email"]) else "그 밖의 언어"
    print(f"      [자식 A: detect] 언어 = {lang}")
    return {}


def classify_shared(state: SharedState) -> dict:
    """공용 상태의 email을 읽어 환불·배송·기타 중 하나로 분류한다."""
    res = llm.invoke([
        SystemMessage("고객 메일을 환불, 배송, 기타 중 하나로 분류한다. 다른 말 없이 단어 하나만 답한다."),
        HumanMessage(state["email"]),
    ])
    return {"category": res.content.strip()}


def classify_private(state: SubState) -> dict:
    """자식 B 상태의 text를 읽어 환불·배송·기타 중 하나로 분류한다."""
    res = llm.invoke([
        SystemMessage("고객 메일을 환불, 배송, 기타 중 하나로 분류한다. 다른 말 없이 단어 하나만 답한다."),
        HumanMessage(state["text"]),
    ])
    return {"label": res.content.strip()}


def handle(state: SharedState) -> dict:
    """분류 결과에 맞는 담당자에게 배정한다 (여기서는 배정 메시지 작성으로 대신한다)."""
    return {"handled": f"{state['category']} 담당자에게 배정"}

### 단계 ③ — 그래프 빌더 생성과 노드 등록 (요구사항 4, 5)

자식 그래프는 부모의 노드 부품입니다. 부모에 등록하려면 먼저 자식이 컴파일되어 있어야 하므로, 이 단계는 두 셀로 나뉩니다. ③-a에서 자식 그래프 두 개를 세워 컴파일하고, ③-b에서 부모 그래프 두 개를 열어 노드를 등록합니다.

#### 단계 ③-a — 자식 그래프 구성과 컴파일 (요구사항 4)

자식 그래프도 상태를 넘겨 빌더를 열고, 노드를 등록하고, 엣지를 추가하고, 컴파일합니다. 부모 그래프를 세우는 절차와 같습니다. 컴파일된 자식은 단독으로 `invoke`할 수 있는 완성된 그래프입니다.

In [4]:
child_a = StateGraph(SharedState)          # 방식 A: 부모와 같은 상태 스키마
child_a.add_node("detect", detect)
child_a.add_node("classify", classify_shared)
child_a.add_edge(START, "detect")
child_a.add_edge("detect", "classify")
child_a.add_edge("classify", END)
CHILD_A = child_a.compile()

child_b = StateGraph(SubState)             # 방식 B: 자식 전용 상태 스키마
child_b.add_node("classify", classify_private)
child_b.add_edge(START, "classify")
child_b.add_edge("classify", END)
CHILD_B = child_b.compile()

print("자식 A의 노드:", list(child_a.nodes), "/ 자식 B의 노드:", list(child_b.nodes))

자식 A의 노드: ['detect', 'classify'] / 자식 B의 노드: ['classify']


#### 단계 ③-b — 부모 그래프 빌더 생성과 노드 등록 (요구사항 5)

- 방식 A: `add_node`의 두 번째 인자에 파이썬 함수 대신 컴파일된 자식 그래프를 넣습니다. 부모가 붙이는 이름 「classifier」는 자식 안의 노드 이름과 무관합니다.
- 방식 B: 자식 B의 키 이름이 부모와 다르므로, 부모 상태를 자식 상태로 번역해 넣고 결과를 부모 키로 되받는 함수를 만들어 그 함수를 노드로 등록합니다.

In [5]:
def call_child_b(state: SharedState) -> dict:
    """부모 상태를 자식 B 상태로 번역해 넣고, 결과의 키 이름을 부모 키로 되돌린다."""
    out = CHILD_B.invoke({"text": state["email"]})   # email -> text
    print(f"      [자식 B] label = {out['label']!r}")
    return {"category": out["label"]}                # label -> category


parent_a = StateGraph(SharedState)
parent_a.add_node("classifier", CHILD_A)        # 컴파일된 자식 그래프를 노드로 직접 등록한다
parent_a.add_node("handle", handle)

parent_b = StateGraph(SharedState)
parent_b.add_node("classifier", call_child_b)   # 자식의 래퍼 함수를 노드로 등록한다
parent_b.add_node("handle", handle)

print("부모 A의 노드:", list(parent_a.nodes), "/ 부모 B의 노드:", list(parent_b.nodes))

부모 A의 노드: ['classifier', 'handle'] / 부모 B의 노드: ['classifier', 'handle']


### 단계 ④ — 엣지 연결 (요구사항 6)

두 부모의 연결은 같습니다. classifier 노드에 무엇을 등록했는지는 연결에 나타나지 않습니다.

In [6]:
parent_a.add_edge(START, "classifier")
parent_a.add_edge("classifier", "handle")
parent_a.add_edge("handle", END)

parent_b.add_edge(START, "classifier")
parent_b.add_edge("classifier", "handle")
parent_b.add_edge("handle", END)

print("두 부모 그래프의 연결을 마쳤습니다.")

두 부모 그래프의 연결을 마쳤습니다.


### 단계 ⑤ — 컴파일과 실행 (요구사항 7)

자식 A를 단독으로 먼저 실행해, 자식이 그 자체로 완결된 그래프임을 확인합니다. 그 뒤 같은 메일을 두 부모에 넣습니다. `stream`은 부모의 노드가 하나 끝날 때마다 바뀐 키를 내보내므로, 자식 안의 노드 이름은 부모의 실행 결과에 나타나지 않습니다.

In [7]:
EMAIL = "주문한 물건이 아직도 도착하지 않았습니다. 언제 오나요?"

print("=== 자식 A 단독 실행 ===")
out = CHILD_A.invoke({"email": EMAIL})
print(f"  결과: category={out['category']!r}")
print()

GRAPH_A = parent_a.compile()
GRAPH_B = parent_b.compile()

for name, graph in (("방식 A: 상태 스키마 공유", GRAPH_A), ("방식 B: 스키마 비공유", GRAPH_B)):
    print(f"=== {name} ===")
    final = {"email": EMAIL}
    for step in graph.stream({"email": EMAIL}, stream_mode="updates"):
        for node, patch in step.items():
            print(f"  [{node}] -> {patch}")
            final.update(patch)
    print(f"  [최종 상태] category={final['category']!r} handled={final['handled']!r}")
    print()

=== 자식 A 단독 실행 ===
      [자식 A: detect] 언어 = 한국어
  결과: category='배송'

=== 방식 A: 상태 스키마 공유 ===
      [자식 A: detect] 언어 = 한국어
  [classifier] -> {'email': '주문한 물건이 아직도 도착하지 않았습니다. 언제 오나요?', 'category': '배송'}
  [handle] -> {'handled': '배송 담당자에게 배정'}
  [최종 상태] category='배송' handled='배송 담당자에게 배정'

=== 방식 B: 스키마 비공유 ===
      [자식 B] label = '배송'
  [classifier] -> {'category': '배송'}
  [handle] -> {'handled': '배송 담당자에게 배정'}
  [최종 상태] category='배송' handled='배송 담당자에게 배정'



## 7. 실행 결과 확인

위 실행 결과에서 다음 세 가지를 확인합니다.

1. 자식 A 단독 실행에서 `[자식 A: detect]` 줄이 출력되고 분류 결과가 나옵니다. 자식은 부모 없이도 실행되는 완성된 그래프입니다.
2. 방식 A의 실행 결과에는 `[classifier]`와 `[handle]` 두 줄만 출력됩니다. 자식 안의 detect·classify는 classifier 노드 하나로 접혀 보입니다. `[classifier]`가 돌려준 키에 `category`가 들어 있습니다.
3. 방식 B의 실행 결과에서 `[자식 B] label = …` 줄이 `[classifier]` 줄보다 먼저 출력됩니다. 래퍼 함수가 자식을 부르고 label을 category로 키 이름을 되돌려 돌려준 순서입니다. 세 실행의 분류 결과는 모두 같습니다.